![alt text](./Cerny_logo_1.jpg)

# Analysis of Cerny ventilation recordings

## Further processing of HFOV data for cases `AT000001-AT001251`

This Notebook produces Excel statistics about ventilator parameters in cases ventilated with HFOV. It also exports barplots showing statistics on ventilation modes used. It does not produse graphs on individual recordings . 

### 1. Import the necessary libraries and set options

In [1]:
import IPython
import pandas as pd
import numpy as np
import scipy as sp
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.dates as dates

import os
import sys
import pickle

from pandas import Series, DataFrame
from datetime import datetime, timedelta
from scipy import stats

%matplotlib inline
matplotlib.style.use('classic')
matplotlib.rcParams['figure.facecolor'] = 'w'

pd.set_option('display.max_rows', 100)
pd.set_option('display.max_columns', 300)
pd.set_option('mode.chained_assignment', None) 

In [2]:
print("Python version: {}".format(sys.version))
print("pandas version: {}".format(pd.__version__))
print("matplotlib version: {}".format(matplotlib.__version__))
print("NumPy version: {}".format(np.__version__))
print("IPython version: {}".format(IPython.__version__))

Python version: 3.11.7 (main, Dec 15 2023, 12:09:04) [Clang 14.0.6 ]
pandas version: 2.1.4
matplotlib version: 3.8.0
NumPy version: 1.26.3
IPython version: 8.20.0


In [3]:
# Define styling of boxplots
medianprops = {'color':'black', 'linewidth':2}; boxprops = {'color':'black', 'linestyle':'-'}
whiskerprops = {'color':'black', 'linestyle':'-'}; capprops = {'color':'black', 'linestyle':'-'}
flierprops = {'color':'black', 'marker': '.'}
meanprops =  {'marker':'D', 'markeredgecolor':'black', 'markerfacecolor':'black'}

### 2. List and set the working directory and the directory to write out data

In [4]:
# Topic of the Notebook which will also be the name of the subfolder containing results
TOPIC = 'HFOV'

# Path to project folder containing clinical data (current weights only) and for export of results
PATH = os.path.join(os.sep, 'Users', 'guszti', 'Library', 'Mobile Documents', 'com~apple~CloudDocs', 
                            'Documents', 'Research', 'Ventilation')

# Name of the external hard drive
DRIVE = 'GUSZTI'

# Folder containing the file with the manually collected clinical data
DIR_READ_CLIN = os.path.join(os.sep, PATH, 'ventilation_fabian_new')

# Processed data loaded from external drive
DIR_READ = os.path.join(os.sep, 'Volumes', DRIVE, 'data_dump', 'fabian_new')

# Results will be written in this folder
DIR_WRITE =  os.path.join(os.sep, PATH, 'ventilation_fabian_new', 'Analyses', TOPIC)
os.makedirs(DIR_WRITE, exist_ok=True)

# Images and raw data will be written on an external hard drive
DATA_DUMP = os.path.join(os.sep, 'Volumes', DRIVE, 'data_dump', 'fabian_new', TOPIC)
os.makedirs(DATA_DUMP, exist_ok=True)

In [5]:
DIR_READ_CLIN, DIR_READ, DIR_WRITE, DATA_DUMP,

('/Users/guszti/Library/Mobile Documents/com~apple~CloudDocs/Documents/Research/Ventilation/ventilation_fabian_new',
 '/Volumes/GUSZTI/data_dump/fabian_new',
 '/Users/guszti/Library/Mobile Documents/com~apple~CloudDocs/Documents/Research/Ventilation/ventilation_fabian_new/Analyses/HFOV',
 '/Volumes/GUSZTI/data_dump/fabian_new/HFOV')

### 3. Import processed ventilator data

This recordings all have >10 minutes of HFOV data

In [6]:
# These data include other forms of respiratory support if it was part of the recording
with open(os.path.join(DATA_DUMP, 'data_pars_measurements_hfov.pickle'), 'rb') as handle:
    data_pars_measurements_hfov = pickle.load(handle)
with open(os.path.join(DATA_DUMP, 'data_pars_settings_hfov.pickle'), 'rb') as handle:
    data_pars_settings_hfov = pickle.load(handle)
with open(os.path.join(DATA_DUMP, 'data_pars_alarms_hfov.pickle'), 'rb') as handle:
    data_pars_alarms_hfov = pickle.load(handle)

# These data only include HFOV
with open(os.path.join(DATA_DUMP, 'data_pars_measurements_hfov_only.pickle'), 'rb') as handle:
    data_pars_measurements_hfov_only = pickle.load(handle)
with open(os.path.join(DATA_DUMP, 'data_pars_settings_hfov_only.pickle'), 'rb') as handle:
    data_pars_settings_hfov_only = pickle.load(handle)
with open(os.path.join(DATA_DUMP, 'data_pars_alarms_hfov_only.pickle'), 'rb') as handle:
    data_pars_alarms_hfov_only = pickle.load(handle)

In [7]:
[len(data_pars_measurements_hfov), len(data_pars_settings_hfov), len(data_pars_alarms_hfov),
len(data_pars_measurements_hfov_only), len(data_pars_settings_hfov_only), len(data_pars_alarms_hfov_only)]

[51, 51, 51, 51, 51, 51]

### 4. Combine ventilator parameters and settings

In [8]:
# Which recordings were exclusively HFOV
recordings_exclusively_hfov = []
recordings_hfov_plus_other = []

for case in data_pars_settings_hfov:
    if all(data_pars_settings_hfov[case]['Ventilator_mode'] == 'HFO'):
        recordings_exclusively_hfov.append(case)
    else :
        recordings_hfov_plus_other.append(case)

print(f'Only HFOV: {len(recordings_exclusively_hfov)}','\n',f'HFOV and other mode: {len(recordings_hfov_plus_other)}')

Only HFOV: 38 
 HFOV and other mode: 13


In [9]:
# All recordings included in the study
recordings_included_all = sorted(data_pars_measurements_hfov.keys())
print(f'All recordings included: {len(recordings_included_all)}')

All recordings included: 51


In [10]:
data_pars_hfov = {}
for case in data_pars_measurements_hfov:
    data_pars_hfov[case] = pd.concat([data_pars_measurements_hfov[case], data_pars_settings_hfov[case]], axis=1)
data_pars_hfov_combined = pd.concat(data_pars_hfov)
data_pars_hfov_combined.index.set_names(['case', 'datetime'], inplace=True)
data_pars_hfov_combined.groupby(level=0).size();


In [11]:
data_pars_hfov_only = {}
for case in data_pars_measurements_hfov_only:
    data_pars_hfov_only[case] = pd.concat([data_pars_measurements_hfov_only[case], data_pars_settings_hfov_only[case]], axis=1)
    data_pars_hfov_only_combined = pd.concat(data_pars_hfov_only)
data_pars_hfov_only_combined.index.set_names(['case', 'datetime'], inplace=True)
data_pars_hfov_only_combined.groupby(level=0).size();

### 5. identify recordings and recording parts with HFOV-VG and without HFOV-VG

In [12]:
data_pars_hfov_vg_only = {}
for case in data_pars_hfov_only:
    dta = data_pars_hfov_only[case]
    # Limit data to HFOV-VG periods
    dta = dta[dta['VG_state'] == 'on']
    # Only include if there is HFOV-VG in the recording
    if not dta.empty :
        data_pars_hfov_vg_only[case] = dta

data_pars_hfov_vg_only_combined = pd.concat(data_pars_hfov_vg_only)

# The number of recordings that contains HFOV-VG
recordings_with_hfov_vg = sorted(data_pars_hfov_vg_only.keys())
print(f'Recordings with HFOV-VG: {len(recordings_with_hfov_vg)}')

Recordings with HFOV-VG: 29


In [13]:
data_pars_hfov_vg_only_combined.groupby(level=0).size();

In [14]:
data_pars_hfov_novg_only = {}
for case in data_pars_hfov_only:
    dta = data_pars_hfov_only[case]
    # Limit data to HFOV-VG periods
    dta = dta[dta['VG_state'] == 'off']
    # Only include if there is HFOV without VG in the recording
    if not dta.empty :
        data_pars_hfov_novg_only[case] = dta

data_pars_hfov_novg_only_combined = pd.concat(data_pars_hfov_novg_only)

# The number of recordings that contains part of HFOV without VG
recordings_with_hfov_novg = sorted(data_pars_hfov_novg_only.keys())
print(f'Recordings with HFOV without VG: {len(recordings_with_hfov_novg)}')

Recordings with HFOV without VG: 37


In [15]:
data_pars_hfov_novg_only_combined.groupby(level=0).size();

In [16]:
recording_duration_hfov = pd.concat([data_pars_hfov_combined.groupby(level=0).size(), data_pars_hfov_only_combined.groupby(level=0).size(),
                                     data_pars_hfov_vg_only_combined.groupby(level=0).size(),
                                     data_pars_hfov_novg_only_combined.groupby(level=0).size()], axis=1)
recording_duration_hfov.columns = ['whole_recording', 'HFOV', 'HFOV-VG', 'HFOV-noVG']
# Replace NaN (where there was HFOV-VG) with 0
recording_duration_hfov = recording_duration_hfov.fillna(0)
# To get recording duration in seconds, multiply it with 2, as sampling rate was 0.5 Hz
recording_duration_hfov = recording_duration_hfov * 2
recording_duration_hfov.head()

,whole_recording,HFOV,HFOV-VG,HFOV-noVG
AT000033,2430,2430,2424.0,6.0
AT000034,1654,1654,1654.0,0.0
AT000048,2844,2844,0.0,2844.0
AT000086,2042,2042,0.0,2042.0
AT000122,2340,2340,1594.0,746.0


In [17]:
# Only consider recordings as HFOV-VG or HFOV-noVG if they comprise at least 90% within the HFOV part
recordings_with_hfov_vg = [rec for rec in recordings_with_hfov_vg if recording_duration_hfov.loc[rec]['HFOV-VG'] >= 
                                                                     recording_duration_hfov.loc[rec]['HFOV'] * 0.9]
recordings_with_hfov_novg = [rec for rec in recordings_with_hfov_novg if recording_duration_hfov.loc[rec]['HFOV-noVG'] >= 
                                                                         recording_duration_hfov.loc[rec]['HFOV'] * 0.9]
recordings_with_both_hfov_vg_novg = sorted(set(recording_duration_hfov.index) - 
                                           set(recordings_with_hfov_vg) - set(recordings_with_hfov_novg))

In [18]:
len(recordings_with_hfov_vg), len(recordings_with_hfov_novg), len(recordings_with_both_hfov_vg_novg)

(26, 22, 3)

In [19]:
recording_duration_hfov_vg = recording_duration_hfov.loc[recordings_with_hfov_vg]
recording_duration_hfov_novg = recording_duration_hfov.loc[recordings_with_hfov_novg]
recording_duration_both_hfov_vg_novg = recording_duration_hfov.loc[recordings_with_both_hfov_vg_novg]

In [20]:
recording_duration_hfov_vg.head(2)

,whole_recording,HFOV,HFOV-VG,HFOV-noVG
AT000033,2430,2430,2424.0,6.0
AT000034,1654,1654,1654.0,0.0


In [21]:
recording_duration_hfov_novg.head(2)

,whole_recording,HFOV,HFOV-VG,HFOV-noVG
AT000048,2844,2844,0.0,2844.0
AT000086,2042,2042,0.0,2042.0


In [22]:
recording_duration_both_hfov_vg_novg

,whole_recording,HFOV,HFOV-VG,HFOV-noVG
AT000122,2340,2340,1594.0,746.0
AT000423,5474,5474,2864.0,2610.0
AT000653,2312,2312,1582.0,730.0


In [23]:
writer = pd.ExcelWriter(os.path.join(DIR_WRITE, 'recording_duration_hfov.xlsx'))
recording_duration_hfov.to_excel(writer, 'hfov_all')
recording_duration_hfov_vg.to_excel(writer, 'hfov_vg')
recording_duration_hfov_novg.to_excel(writer, 'hfov_novg')
recording_duration_both_hfov_vg_novg.to_excel(writer, 'both_hfov_vg_novg')
writer.close()

In [24]:
# Limit ventilator data to the included HFOV-VG and noVG recordings
data_pars_hfov_vg_only = {recording : data for recording, data in data_pars_hfov_vg_only.items() 
                          if recording in recordings_with_hfov_vg}
data_pars_hfov_novg_only = {recording : data for recording, data in data_pars_hfov_novg_only.items() 
                            if recording in recordings_with_hfov_novg}
data_pars_both_hfov_vg_novg = {recording : data for recording, data in data_pars_hfov_only.items() 
                            if recording in recordings_with_both_hfov_vg_novg}

len(data_pars_hfov_vg_only), len(data_pars_hfov_novg_only), len(data_pars_both_hfov_vg_novg)

(26, 22, 3)

In [25]:
data_pars_hfov_vg_only_combined = pd.concat(data_pars_hfov_vg_only)
data_pars_hfov_novg_only_combined = pd.concat(data_pars_hfov_novg_only)
data_pars_both_hfov_vg_novg_combined = pd.concat(data_pars_both_hfov_vg_novg)

### 6. Export processed ventilator data

In [26]:
[len(data_pars_hfov), len(data_pars_hfov_only), len(data_pars_hfov_vg_only), len(data_pars_hfov_novg_only), 
 len(data_pars_both_hfov_vg_novg)]

[51, 51, 26, 22, 3]

In [27]:
[len(data_pars_hfov_combined), len(data_pars_hfov_only_combined), len(data_pars_hfov_vg_only_combined), 
 len(data_pars_hfov_novg_only_combined), len(data_pars_both_hfov_vg_novg_combined)]

[89132, 80961, 42108, 33157, 5063]

In [28]:
%%time

## Export into pickle archives as dictionaries DataFrames

# All recordings included in the final analysis, both HFOV and conventional ventilation data
with open(os.path.join(DATA_DUMP, 'data_pars_hfov.pickle'), 'wb') as handle:
    pickle.dump(data_pars_hfov, handle, protocol=pickle.HIGHEST_PROTOCOL)    

# All recordings included in the final analysis, only HFOV parts
with open(os.path.join(DATA_DUMP, 'data_pars_hfov_only.pickle'), 'wb') as handle:
    pickle.dump(data_pars_hfov_only, handle, protocol=pickle.HIGHEST_PROTOCOL)

# Only recordings selected as HFOV-VG and only HFOV-VG parts
with open(os.path.join(DATA_DUMP, 'data_pars_hfov_vg_only.pickle'), 'wb') as handle:
    pickle.dump(data_pars_hfov_vg_only, handle, protocol=pickle.HIGHEST_PROTOCOL)

# Only recordings selected as HFOV-noVG and only HFOV-noVG parts
with open(os.path.join(DATA_DUMP, 'data_pars_hfov_novg_only.pickle'), 'wb') as handle:
    pickle.dump(data_pars_hfov_novg_only, handle, protocol=pickle.HIGHEST_PROTOCOL)

# Only recordings selected as containing >10% of both HFOV-VG and HFOV-noVG, and only HFOV parts
with open(os.path.join(DATA_DUMP, 'data_pars_both_hfov_vg_novg.pickle'), 'wb') as handle:
    pickle.dump(data_pars_both_hfov_vg_novg, handle, protocol=pickle.HIGHEST_PROTOCOL)


## Export into pickle archive as combined in a single Dataframe

# All recordings included in the final analysis, both HFOV and conventional ventilation data
with open(os.path.join(DATA_DUMP, 'data_pars_hfov_combined.pickle'), 'wb') as handle:
    pickle.dump(data_pars_hfov_combined, handle, protocol=pickle.HIGHEST_PROTOCOL)    

# All recordings included in the final analysis, only HFOV parts
with open(os.path.join(DATA_DUMP, 'data_pars_hfov_only_combined.pickle'), 'wb') as handle:
    pickle.dump(data_pars_hfov_only_combined, handle, protocol=pickle.HIGHEST_PROTOCOL)

# Only recordings selected as HFOV-VG and only HFOV-VG parts, combined in a single DataFrame
with open(os.path.join(DATA_DUMP, 'data_pars_hfov_vg_only_combined.pickle'), 'wb') as handle:
    pickle.dump(data_pars_hfov_vg_only_combined, handle, protocol=pickle.HIGHEST_PROTOCOL)

# Only recordings selected as HFOV-noVG and only HFOV-noVG parts
with open(os.path.join(DATA_DUMP, 'data_pars_hfov_novg_only_combined.pickle'), 'wb') as handle:
    pickle.dump(data_pars_hfov_novg_only_combined, handle, protocol=pickle.HIGHEST_PROTOCOL)

# Only recordings selected as containing >10% of both HFOV-VG and HFOV-noVG, and only HFOV parts
with open(os.path.join(DATA_DUMP, 'data_pars_both_hfov_vg_novg_combined.pickle'), 'wb') as handle:
    pickle.dump(data_pars_both_hfov_vg_novg_combined, handle, protocol=pickle.HIGHEST_PROTOCOL)

CPU times: user 120 ms, sys: 75.1 ms, total: 195 ms
Wall time: 542 ms
